# Notebook 99: Ad-Hoc Analyses
Checks run to answer one specific question, almost always *"would this change the conclusion?"*, which are not part of the manuscript's analysis plan. Each gets its own section: the question, the comparison that answers it, and what it settled.
<br><br>
Nothing here is a reported result. A section either confirms that a reported model is robust to the choice it interrogates, in which case the reported model stands unchanged and the section is kept only so the answer exists if a reviewer asks; or it does not, in which case the finding moves into the notebook that owns that analysis and this section goes away.
<br><br>
These checks are deliberately kept out of notebooks 7 to 11, so that each of those stays the analysis the manuscript describes, with one specification ladder and one story.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.power import TTestIndPower

from helpers.config import *
from helpers import dataset
from helpers.equivalence import LOG_BOUND, tost_coefficient, tost_from_estimate
from helpers.r_models import fit_glmm

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel. The two outcome frames are built exactly as in notebooks 7 and 8.

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)
FWCI_DF = dataset.to_smf_columns(
    pd.concat([FEATURES_DF, np.log(dataset.fwci_with_zeros_imputed(combined))], axis=1)
    .rename(columns={"fwci_with_zeros_imputed": "log(Imputed FWCI)"})
    .drop(columns=["log(Weeks Since Pub.)"])
)

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Last-Author (Lab) Clustering
Every model in notebooks 7 and 8 treats the **venue** as the only source of dependence between articles: Model 0 clusters its standard errors by venue, and Models 2 and 3 give the venue a random intercept. But sharing data is plausibly a *lab* habit rather than a per-article choice, and notebook 2 reports that **105 of the 234 articles (44.9%) share a last author** with another article in the sample, one last author contributing seven. If articles from the same lab resemble each other, the standard error of the sharing coefficient is too small, and the null result is less safe than it looks.
<br><br>
The question is narrow: does adding last-author dependence move the sharing estimate, or its standard error, enough to matter? We ask it at both ends of the ladder, Model 0 (no venue term, cluster-robust standard errors) and Model 3 (venue impact plus a venue random intercept), for both outcomes.
<br><br>
The last author stands in for the lab. It is matched on a name string, with the same four manually validated merges notebook 2 applies, so it **under**-merges rather than over-merges: if anything the clusters are too small, which works against finding anything here.

In [3]:
# the last-author extraction of notebook 2, with its manually validated merges of near-duplicate names
_LAST_AUTHOR_MERGES = {"Becker, S": "Becker, SI", "Hooge, I": "Hooge, ITC", "Jiang, YV": "Jiang, YHV", "Zelisky, G": "Zelinsky, GJ"}

last_author = (
    combined["Authors"]
    .map(lambda names: names.split("; ")[-1])
    .replace(_LAST_AUTHOR_MERGES)
    .rename("last_author")
)
CITATIONS_DF["last_author"] = last_author
FWCI_DF["last_author"] = last_author


def cluster_structure(labels: pd.Series) -> dict[str, float]:
    """How coarse a grouping is: how many clusters, how big, and how much of the sample sits in a cluster
    that can carry any within-cluster correlation at all (a singleton cannot)."""
    sizes = labels.value_counts()
    return {
        "clusters": len(sizes),
        "mean size": sizes.mean(),
        "max size": int(sizes.max()),
        "singletons": int((sizes == 1).sum()),
        "% of articles in a cluster of 2+": 100 * sizes[sizes > 1].sum() / len(labels),
    }


pd.DataFrame({
    "last author": cluster_structure(last_author),
    "venue (as reported)": cluster_structure(combined["VenueName"]),
}).T.round(2)

,clusters,mean size,max size,singletons,% of articles in a cluster of 2+
last author,167.0,1.40,7.0,129.0,44.87
venue (as reported),23.0,10.17,43.0,7.0,97.01


### Model 0 :: Which Clustering for the Standard Errors?
Clustering changes only the variance, never the point estimate, so `b` is identical in all three rows and only the standard error is informative. The two-way row clusters on venue and last author at once, which is the conservative choice when neither grouping is nested in the other. The last three columns are a TOST of the sharing coefficient within ±0.3 on the log scale (the smallest effect of interest of notebooks 7 and 8; a *t* reference on the number of clusters minus one), with the narrowest bound its 90% interval supports.

In [4]:
SHARING_TERM = "is_sharing_data"
COVARIATES = "has_us_author + is_open_access + has_preprint + log_number_of_authors"
# FWCI is normalized by publication year, so it takes no age term (see notebook 8)
OUTCOMES = {
    "citations": (CITATIONS_DF, "log_citations", f"{COVARIATES} + log_weeks_since_pub"),
    "FWCI": (FWCI_DF, "log_imputed_fwci", COVARIATES),
}


def clustered_ols(data: pd.DataFrame, outcome: str, covariates: str, groups: np.ndarray) -> dict[str, float]:
    """Model 0 of notebooks 7-8: OLS with no venue term and cluster-robust standard errors
    (CR1, with a t reference on G - 1 degrees of freedom)."""
    fit = smf.ols(f"{outcome} ~ {SHARING_TERM} + {covariates}", data=data).fit(
        cov_type="cluster", cov_kwds={"groups": groups, "use_correction": True}, use_t=True,
    )
    ci_low, ci_high = fit.conf_int().loc[SHARING_TERM]
    tost = tost_coefficient(fit, SHARING_TERM)
    return {
        "b": fit.params[SHARING_TERM], "SE": fit.bse[SHARING_TERM], "p": fit.pvalues[SHARING_TERM],
        "CI low": ci_low, "CI high": ci_high, "clusters": len(np.unique(groups, axis=0)),
        "TOST p": tost.p_tost, "outcome": tost.outcome, "smallest bound": tost.smallest_bound,
    }


model0_rows = {}
for outcome_label, (data, outcome, covariates) in OUTCOMES.items():
    venues, authors = pd.factorize(data["venue_name"])[0], pd.factorize(data["last_author"])[0]
    clusterings = {
        "venue (as reported)": venues,
        "last author": authors,
        "two-way: venue + last author": np.column_stack([venues, authors]),
    }
    for clustering, groups in clusterings.items():
        model0_rows[(outcome_label, clustering)] = clustered_ols(data, outcome, covariates, groups)

model0_comparison = pd.DataFrame(model0_rows).T.infer_objects()
model0_comparison.index.names = ["Outcome", "Clustering"]
model0_comparison.round(4)

b      SE       p  CI low  \
Outcome   Clustering                                                     
citations venue (as reported)           0.1370  0.0946  0.1620 -0.0593   
          last author                   0.1370  0.1284  0.2877 -0.1166   
          two-way: venue + last author  0.1370  0.1014  0.1904 -0.0733   
FWCI      venue (as reported)           0.1716  0.1282  0.1945 -0.0943   
          last author                   0.1716  0.1559  0.2726 -0.1362   
          two-way: venue + last author  0.1716  0.1286  0.1958 -0.0951   

                                        CI high  clusters  TOST p  \
Outcome   Clustering                                                
citations venue (as reported)            0.3332        23  0.0495   
          last author                    0.3905       167  0.1030   
          two-way: venue + last author   0.3472       217  0.0610   
FWCI      venue (as reported)            0.4374        23  0.1636   
          last author                    0.4793       167  0.2055   
          two-way: venue + last author   0.4382       217  0.1644   

                                             outcome  smallest bound  
Outcome   Clustering                                                  
citations venue (as reported)             equivalent          0.2995  
          last author                   inconclusive          0.3493  
          two-way: venue + last author  inconclusive          0.3110  
FWCI      venue (as reported)           inconclusive          0.3917  
          last author                   inconclusive          0.4294  
          two-way: venue + last author  inconclusive          0.3924

### Model 3 :: Does a Last-Author Random Intercept Earn Its Place?
Here the last author enters as a second random intercept, **crossed** with the venue rather than nested in it (a lab publishes in several journals, and a journal hosts several labs). `statsmodels` cannot fit crossed random effects, so these go through the R bridge (`lme4`), which also means they are fitted by **ML** rather than the REML of notebooks 7 and 8: the "venue (as reported)" row is therefore close to, but not identical with, the Model 3 those notebooks report. ML is what makes the three structures' likelihoods comparable, which is the point of the table.
<br><br>
A variance component is tested on the boundary of its parameter space, so the likelihood-ratio statistic is referred to a 50:50 mixture of $\chi^2(0)$ and $\chi^2(1)$, which is half the ordinary one-degree-of-freedom p-value.<br><br>The table also carries the same TOST within ±0.3 (a normal reference, since the R fit does not return degrees of freedom).

In [5]:
RANDOM_STRUCTURES = {
    "venue (as reported)": "(1 | venue_name)",
    "venue + last author": "(1 | venue_name) + (1 | last_author)",
    "last author only": "(1 | last_author)",
}


def crossed_lme(data: pd.DataFrame, outcome: str, covariates: str, random_effects: str) -> dict[str, float]:
    """Model 3 of notebooks 7-8 (venue impact as a covariate, venue as a random intercept), with the random
    part replaced by `random_effects`. Fitted in R by ML, so that the structures' likelihoods are comparable."""
    fit = fit_glmm(
        data, f"{outcome} ~ {SHARING_TERM} + {covariates} + venue_impact + {random_effects}",
        "gaussian", factors=("venue_name", "last_author"),
    )
    effect = fit.fixed_effects.loc[SHARING_TERM]
    variances = dict(zip(fit.variance_components["group"], fit.variance_components["variance"]))
    total = sum(variances.values())
    tost = tost_from_estimate(effect["estimate"], effect["se"], -LOG_BOUND, LOG_BOUND)
    return {
        "b": effect["estimate"], "SE": effect["se"], "p": effect["p"],
        "TOST p": tost.p_tost, "outcome": tost.outcome, "smallest bound": tost.smallest_bound,
        "venue var": variances.get("venue_name", np.nan), "author var": variances.get("last_author", np.nan),
        "residual": variances["Residual"],
        "venue ICC": variances.get("venue_name", np.nan) / total,
        "author ICC": variances.get("last_author", np.nan) / total,
        "logLik": fit.fit["log_lik"], "AIC": fit.fit["aic"],
        "converged": fit.fit["converged"], "singular": fit.fit["singular"],
    }


model3_rows = {
    (outcome_label, structure): crossed_lme(data, outcome, covariates, random_effects)
    for outcome_label, (data, outcome, covariates) in OUTCOMES.items()
    for structure, random_effects in RANDOM_STRUCTURES.items()
}
model3_comparison = pd.DataFrame(model3_rows).T
model3_comparison.index.names = ["Outcome", "Random effects"]
model3_comparison.round(4)

b        SE         p    TOST p  \
Outcome   Random effects                                                
citations venue (as reported)  0.043753  0.127567  0.731937  0.022283   
          venue + last author   0.06641   0.12633  0.599631  0.032226   
          last author only     0.096813  0.123791  0.434964   0.05036   
FWCI      venue (as reported)  0.071671  0.150358   0.63406  0.064436   
          venue + last author  0.116362  0.148786  0.434994  0.108557   
          last author only     0.138118  0.146748  0.347581  0.134986   

                                    outcome smallest bound venue var  \
Outcome   Random effects                                               
citations venue (as reported)    equivalent       0.253582  0.052596   
          venue + last author    equivalent       0.274205  0.037959   
          last author only     inconclusive       0.300431       NaN   
FWCI      venue (as reported)  inconclusive       0.318989  0.057479   
          venue + last author  inconclusive       0.361094  0.050653   
          last author only     inconclusive       0.379497       NaN   

                              author var  residual venue ICC author ICC  \
Outcome   Random effects                                                  
citations venue (as reported)        NaN  0.629359  0.077125        NaN   
          venue + last author   0.191234   0.44932  0.055944   0.281843   
          last author only      0.220219  0.463076       NaN    0.32229   
FWCI      venue (as reported)        NaN  0.928048  0.058323        NaN   
          venue + last author   0.318951  0.626881  0.050832   0.320076   
          last author only       0.34464  0.660209       NaN   0.342977   

                                   logLik         AIC converged singular  
Outcome   Random effects                                                  
citations venue (as reported)  -283.81429   587.62858      True    False  
          venue + last author -280.443941  582.887882      True    False  
          last author only    -282.538824  585.077648      True    False  
FWCI      venue (as reported) -328.121555  674.243111      True    False  
          venue + last author -324.611171  669.222343      True    False  
          last author only    -327.025778  672.051557      True    False

### Is the Added Component Supported, and Does the Estimate Move?
Two questions, one table. `chi2` and its mixture p-value ask whether the last-author variance is there at all. `b shift` and `SE ratio` ask the question this section exists for: whether accounting for it changes what the manuscript would report.

In [6]:
summary_rows = {}
for outcome_label in OUTCOMES:
    reported = model3_comparison.loc[(outcome_label, "venue (as reported)")]
    with_author = model3_comparison.loc[(outcome_label, "venue + last author")]
    statistic = 2 * (with_author["logLik"] - reported["logLik"])
    summary_rows[outcome_label] = {
        "chi2 (boundary, 1 df)": statistic,
        "p (50:50 mixture)": 0.5 * stats.chi2.sf(max(statistic, 0.0), df=1),
        "author ICC": with_author["author ICC"],
        "venue ICC": with_author["venue ICC"],
        "% residual removed": 100 * (1 - with_author["residual"] / reported["residual"]),
        "b (reported)": reported["b"],
        "b (+ author)": with_author["b"],
        "b shift": with_author["b"] - reported["b"],
        "SE ratio": with_author["SE"] / reported["SE"],
        "p (reported)": reported["p"],
        "p (+ author)": with_author["p"],
    }

author_variance_summary = pd.DataFrame(summary_rows).T
author_variance_summary.index.name = "Outcome"
author_variance_summary.round(4)

,"chi2 (boundary, 1 df)",p (50:50 mixture),author ICC,venue ICC,% residual removed,b (reported),b (+ author),b shift,SE ratio,p (reported),p (+ author)
Outcome,,,,,,,,,,,
citations,6.7407,0.0047,0.2818,0.0559,28.6068,0.0438,0.0664,0.0227,0.9903,0.7319,0.5996
FWCI,7.0208,0.0040,0.3201,0.0508,32.4516,0.0717,0.1164,0.0447,0.9895,0.6341,0.4350


## Statistical Sensitivity
The reported models find no citation advantage for data sharing. A null is only informative if the design could have found an effect had one been there, so this section asks what effect this design was in a position to detect.
<br><br>
**This is a sensitivity analysis, not post-hoc power.** It takes the standard error the reported models actually achieved and asks what *true* effect that precision could reject the null for. It never uses the observed effect, which would be circular: an observed null always yields low "post-hoc power" by construction, so computing it adds nothing. What the achieved standard error does carry is everything the design paid for, the covariates and the venue clustering included.
<br><br>
The yardstick is the literature the report cites: citation premiums of roughly **4%** (Colavizza et al., 2024) through **20%** in psychology (Pan et al., 2026) to **25%** over a three-year window (Colavizza et al., 2020). The sharing coefficient is on the log scale, so a premium of *p* corresponds to a coefficient of log(1 + *p*).
<br><br>
Two conventions matter and are reported side by side. The project tests the sharing hypothesis **one-sided**, but the mixed-model tables report **two-sided** *p*-values, which are the more conservative of the two and the ones a reader of Tables 3 and 4 would apply.

In [7]:
PREMIUMS = [0.04, 0.10, 0.20, 0.25]      # the citation premiums reported in the prior literature
ALTERNATIVES = {"one-sided": 0.05, "two-sided": 0.025}      # the alpha in the relevant tail
REPORTED_TERM = "C(is_sharing_data)[T.1]"


def minimum_detectable_effect(se: float, power: float, alpha: float) -> float:
    """The smallest true coefficient a design with this standard error rejects the null for, at `power`."""
    return (stats.norm.isf(alpha) + stats.norm.ppf(power)) * se


def power_for(effect: float, se: float, alpha: float) -> float:
    """The probability of rejecting the null when the true coefficient is `effect`."""
    return float(stats.norm.sf(stats.norm.isf(alpha) - effect / se))


# the two models the manuscript reports (Model 3 of notebooks 7 and 8), refitted here so this section
# stands on the same standard errors the tables print
reported_models = {
    "citations": smf.mixedlm(
        "log_citations ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint)"
        " + log_weeks_since_pub + log_number_of_authors + venue_impact",
        data=CITATIONS_DF, groups=CITATIONS_DF["venue_name"]).fit(reml=True),
    "FWCI": smf.mixedlm(
        "log_imputed_fwci ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint)"
        " + log_number_of_authors + venue_impact",
        data=FWCI_DF, groups=FWCI_DF["venue_name"]).fit(reml=True),
}
STANDARD_ERRORS = {name: float(fit.bse[REPORTED_TERM]) for name, fit in reported_models.items()}

achieved = {}
for name, fit in reported_models.items():
    estimate = float(fit.params[REPORTED_TERM])
    low, high = fit.conf_int().loc[REPORTED_TERM]
    achieved[name] = {
        "b": estimate, "SE": STANDARD_ERRORS[name], "% diff": 100 * np.expm1(estimate),
        "95% CI low (%)": 100 * np.expm1(low), "95% CI high (%)": 100 * np.expm1(high),
    }
print("What the data actually say, as a percentage difference:")
pd.DataFrame(achieved).T.round(2)

What the data actually say, as a percentage difference:


,b,SE,% diff,95% CI low (%),95% CI high (%)
citations,0.04,0.13,3.84,-19.72,34.31
FWCI,0.07,0.15,6.87,-20.96,44.51


In [8]:
mde_rows = {
    (name, alternative, f"{power:.0%} power"): {
        "SE": se,
        "MDE (log)": (effect := minimum_detectable_effect(se, power, alpha)),
        "MDE (%)": 100 * np.expm1(effect),
    }
    for name, se in STANDARD_ERRORS.items()
    for alternative, alpha in ALTERNATIVES.items()
    for power in (0.80, 0.90)
}
mde = pd.DataFrame(mde_rows).T
mde.index.names = ["outcome", "test", "power"]
print("Smallest premium this design could detect:")
mde.round(3)

Smallest premium this design could detect:


SE  MDE (log)  MDE (%)
outcome   test      power                               
citations one-sided 80% power  0.131      0.326   38.603
                    90% power  0.131      0.384   46.844
          two-sided 80% power  0.131      0.368   44.457
                    90% power  0.131      0.426   53.046
FWCI      one-sided 80% power  0.154      0.383   46.628
                    90% power  0.154      0.450   56.901
          two-sided 80% power  0.154      0.431   53.915
                    90% power  0.154      0.499   64.698

In [9]:
power_rows = {
    (name, f"+{premium:.0%}"): {
        "b (log)": np.log1p(premium),
        **{alternative: power_for(np.log1p(premium), se, alpha) for alternative, alpha in ALTERNATIVES.items()},
    }
    for name, se in STANDARD_ERRORS.items()
    for premium in PREMIUMS
}
power_table = pd.DataFrame(power_rows).T
power_table.index.names = ["outcome", "premium"]
print("Power to detect the premiums reported in the literature:")
power_table.round(3)

Power to detect the premiums reported in the literature:


b (log)  one-sided  two-sided
outcome   premium                               
citations +4%        0.039      0.089      0.048
          +10%       0.095      0.179      0.109
          +20%       0.182      0.399      0.284
          +25%       0.223      0.522      0.397
FWCI      +4%        0.039      0.082      0.044
          +10%       0.095      0.153      0.090
          +20%       0.182      0.323      0.219
          +25%       0.223      0.423      0.305

In [10]:
# the univariate view: how large a corpus a premium of this size would need, at the same 2:1 split
sharing_mask = CITATIONS_DF["is_sharing_data"] == 1
sharing_values, non_sharing_values = CITATIONS_DF.loc[sharing_mask, "log_citations"], CITATIONS_DF.loc[~sharing_mask, "log_citations"]
pooled_sd = np.sqrt((sharing_values.var(ddof=1) + non_sharing_values.var(ddof=1)) / 2)
ratio = len(sharing_values) / len(non_sharing_values)
analysis = TTestIndPower()

print(f"pooled SD of log(1 + citations) = {pooled_sd:.4f}, n = {len(non_sharing_values)} non-sharing vs {len(sharing_values)} sharing")
sample_rows = {}
for premium in PREMIUMS:
    d = np.log1p(premium) / pooled_sd
    required = analysis.solve_power(effect_size=d, power=0.80, ratio=ratio, alpha=0.05, alternative="larger")
    sample_rows[f"+{premium:.0%}"] = {
        "Cohen d": d,
        "power at this N (one-sided)": analysis.power(
            effect_size=d, nobs1=len(non_sharing_values), ratio=ratio, alpha=0.05, alternative="larger"),
        "articles needed for 80% power": int(np.ceil(required * (1 + ratio))),
    }
pd.DataFrame(sample_rows).T.round(3)

pooled SD of log(1 + citations) = 0.9145, n = 154 non-sharing vs 80 sharing


,Cohen d,power at this N (one-sided),articles needed for 80% power
+4%,0.043,0.091,14942.0
+10%,0.104,0.187,2532.0
+20%,0.199,0.420,693.0
+25%,0.244,0.548,463.0


### Reading the Sensitivity Analysis
**The design is powered for a premium of roughly 40% to 50%, about twice the largest effect the literature reports.** At 80% power and testing one-sided, the smallest detectable premium is **+39%** for accumulated citations and **+47%** for FWCI; on the two-sided tests the tables actually report, +44% and +54%.
<br><br>
Against the published effects the picture is worse than a coin flip almost everywhere. For the 25% premium of Colavizza et al. (2020) this design has **52%** power on citations and 42% on FWCI, one-sided, falling to 40% and 31% two-sided. For the 20% reported in psychology, 40% and 32%. For the 4% long-term estimate of Colavizza et al. (2024), **9%** and 8%, which is barely above the false-positive rate itself. Univariately the same conclusion arrives from the other direction: detecting a 25% premium at 80% power would take about **463 articles** at this 2:1 split, and a 4% premium about 15,000, against the 234 available here.
<br><br>
The confidence intervals were already saying this. The adjusted estimates are +3.8% (95% CI -20% to +34%) for citations and +6.9% (-21% to +45%) for FWCI, and **both intervals contain +25%**. The study did not fail to replicate the Open Data Citation Advantage; it was never in a position to adjudicate it.
<br><br>
This also explains the equivalence tests rather than sitting beside them. The tightest bound this precision supports is about ±0.25 to ±0.32 on the log scale, that is +29% to +38%, which lies *above* the range being argued about. The prespecified ±0.3 margin was therefore set at roughly the edge of what is resolvable here, which is why citations cleared it and FWCI did not.
<br><br>
**What follows for the write-up.** The null results bound the plausible effect in this subfield rather than refuting the ODCA, and the Limitations section should say so with these numbers rather than with the current qualitative "the sample size limited statistical power". A 234-article audit of one subfield was never going to settle a question answered with 368,000 articles; its contribution is a subfield-specific interval, which is worth reporting even when it is wide.

## Conclusions

### Last-author (lab) clustering
**The dependence is real, it is stronger than the venue's, and it changes nothing about the sharing result.** Both halves matter, and they point in opposite directions.

*The clustering is there.* A crossed last-author intercept is clearly supported: $\chi^2 = 6.74$, $p = 0.005$ for citations and $\chi^2 = 7.02$, $p = 0.004$ for FWCI, against the 50:50 boundary mixture. It absorbs roughly **five times** the variance the venue does, with an author ICC of 0.28 (citations) and 0.32 (FWCI) against a venue ICC of 0.06 and 0.05 once both are in the model, and it removes 29% and 32% of the residual variance. Articles in this corpus cluster more by lab than by journal, which is worth knowing even though nothing in the manuscript turns on it.

*The sharing estimate does not care.* In Model 3 it moves from $b = 0.044$ to $0.066$ (citations) and from $0.072$ to $0.116$ (FWCI), in both cases **away** from zero, while the standard error is unchanged to three decimals ($\times 0.99$); it stays non-significant ($p$ from 0.73 to 0.60, and from 0.63 to 0.44) and, for citations, equivalent to zero within ±0.3 on the log scale (TOST $p$ = 0.022 and 0.032), though not demonstrably so for FWCI (0.064 and 0.109, inconclusive). In Model 0 the coefficient cannot move at all, since clustering touches only the variance, and the author-clustered standard error is the **wider** of the two (0.128 against 0.095 for citations, 0.156 against 0.128 for FWCI). So the venue clustering the manuscript reports is, if anything, the slightly less conservative of the two choices, and the result is non-significant under either. The TOST shows what the wider standard error costs: for citations, Model 0 is equivalent within ±0.3 with the venue clustering ($p$ = 0.0495) but not with the last-author clustering (0.103) or the two-way one (0.061), and none of the FWCI rows is.

*Decision: the notebooks 7-8 ladder stays as it is.* A fifth specification would change no conclusion and would cost a paragraph of the Methods for a model nobody asked about. What this section buys is a prepared answer, should a reviewer ask whether lab-level dependence inflates the confidence in the null: it does not, the component is estimable and large, and the estimate is unchanged with it in the model.

*Caveats.* These mixed fits are ML, while notebooks 7 and 8 fit Models 2 and 3 by REML, so the reference row differs slightly from the reported Model 3 ($b = 0.044$ against 0.038 for citations, 0.072 against 0.067 for FWCI); the comparison within the table is like-for-like, which is what it is for. And the last author is a name-matched proxy for the lab, so the clusters are if anything too fine.

### Statistical sensitivity
**The design can detect a citation premium of about 39% (citations) or 47% (FWCI) at 80% power, one-sided, and roughly 44% and 54% on the two-sided tests the tables report.** Power against the published effects is accordingly low: 52% and 42% for a 25% premium, 9% and 8% for a 4% one. Detecting 25% univariately at 80% power would need about 463 articles rather than 234. Both adjusted confidence intervals (-20% to +34% for citations, -21% to +45% for FWCI) contain +25%.

*Decision: this changes no analysis, but it should change one sentence of the manuscript.* The Limitations paragraph currently notes only that the sample size limited power; it should give the minimum detectable effect, and frame the nulls as bounding the effect in this subfield rather than refuting the Open Data Citation Advantage.